# Supplement — Practical Model Training with PyTorch Layers

**CSS371: Natural Language Processing**  
**Focus:** construct, train, evaluate, and use neural networks through high-level PyTorch APIs

This notebook assumes the gradient concepts have already been studied. We will not derive, inspect, or print gradients. Instead, we use standard framework components:

- `nn.Sequential` and reusable modules;
- `TensorDataset` and `DataLoader`;
- `BCEWithLogitsLoss`;
- `torch.optim.Adam`;
- training, validation, inference, and error analysis.


## Learning outcomes

Students should be able to:

1. define a neural network from PyTorch layers;
2. prepare tensors and mini-batch data loaders;
3. implement a reusable training loop;
4. switch correctly between `model.train()` and `model.eval()`;
5. convert logits into probabilities and class predictions;
6. calculate classification metrics;
7. train a simple sentence classifier using mock data;
8. save and load model parameters.


In [3]:
import sys, random, re
from collections import Counter
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | device={DEVICE}")


Python 3.12.7 | PyTorch 2.13.0+cpu | device=cpu


# Part I — Two-dimensional binary classification

## 1. Generate and package mock data

Class 0 is centered near $(-1,-1)$ and class 1 near $(1,1)$. Each example has two features. We reserve 20% for validation and create mini-batch loaders.


In [5]:
def make_blob_data(n_per_class=120,noise=.8,seed=42):
    rng=np.random.default_rng(seed)
    c0=rng.normal((-1,-1),noise,size=(n_per_class,2))
    c1=rng.normal((1,1),noise,size=(n_per_class,2))
    X=np.vstack([c0,c1]).astype("float32")
    y=np.vstack([np.zeros((n_per_class,1)),np.ones((n_per_class,1))]).astype("float32")
    order=rng.permutation(len(X)); return X[order],y[order]

X,y=make_blob_data(); split=int(.8*len(X))
X_train=torch.tensor(X[:split]); y_train=torch.tensor(y[:split])
X_valid=torch.tensor(X[split:]); y_valid=torch.tensor(y[split:])

generator=torch.Generator().manual_seed(SEED)
train_loader=DataLoader(TensorDataset(X_train,y_train),batch_size=32,shuffle=True,generator=generator)
valid_loader=DataLoader(TensorDataset(X_valid,y_valid),batch_size=64,shuffle=False)
print("train:",X_train.shape,y_train.shape,"validation:",X_valid.shape,y_valid.shape)
print("mini-batches per epoch:",len(train_loader))


train: torch.Size([192, 2]) torch.Size([192, 1]) validation: torch.Size([48, 2]) torch.Size([48, 1])
mini-batches per epoch: 6


## 2. Build the model from layers

The network maps

```text
2 features → 8 hidden units → 4 hidden units → 1 logit
```

The final layer has no sigmoid because `BCEWithLogitsLoss` expects raw logits and handles the binary objective stably. Sigmoid is applied only when probabilities are needed for evaluation or inference.


In [7]:
model=nn.Sequential(
    nn.Linear(2,8),
    nn.ReLU(),
    nn.Linear(8,4),
    nn.ReLU(),
    nn.Linear(4,1),
).to(DEVICE)

loss_function=nn.BCEWithLogitsLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=.01)
print(model)
print("trainable parameters:",sum(p.numel() for p in model.parameters() if p.requires_grad))


Sequential(
  (0): Linear(in_features=2, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=4, bias=True)
  (3): ReLU()
  (4): Linear(in_features=4, out_features=1, bias=True)
)
trainable parameters: 65


## 3. Reusable training and evaluation functions

`model.train()` activates training behavior, including dropout or batch-normalization updates when present. `model.eval()` activates evaluation behavior. `torch.no_grad()` avoids storing training bookkeeping during validation.

The five operational steps inside each training batch are:

```text
clear previous state → forward → loss → backward → optimizer update
```

This notebook treats backward calculation as a framework service and does not inspect its internal gradients.


In [9]:
def train_one_epoch(model,loader,loss_function,optimizer,device):
    model.train(); total_loss=0.0; total_correct=0; total_examples=0
    for features,labels in loader:
        features,labels=features.to(device),labels.to(device)
        optimizer.zero_grad()
        logits=model(features)
        loss=loss_function(logits,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()*len(features)
        total_correct+=((torch.sigmoid(logits)>=.5)==labels.bool()).sum().item()
        total_examples+=len(features)
    return total_loss/total_examples,total_correct/total_examples

@torch.no_grad()
def evaluate(model,loader,loss_function,device):
    model.eval(); total_loss=0.0; probabilities=[]; labels_all=[]
    for features,labels in loader:
        features,labels=features.to(device),labels.to(device)
        logits=model(features); loss=loss_function(logits,labels)
        total_loss+=loss.item()*len(features)
        probabilities.append(torch.sigmoid(logits).cpu()); labels_all.append(labels.cpu())
    probabilities=torch.cat(probabilities); labels_all=torch.cat(labels_all)
    accuracy=((probabilities>=.5)==labels_all.bool()).float().mean().item()
    return total_loss/len(labels_all),accuracy,probabilities,labels_all


## 4. Train and select by validation loss

We record training and validation histories. In larger projects, an untouched test set should be used once after model selection; repeatedly consulting test results would leak evaluation information.


In [11]:
history=[]; best_validation=float("inf"); best_state=None
for epoch in range(1,101):
    train_loss,train_accuracy=train_one_epoch(model,train_loader,loss_function,optimizer,DEVICE)
    valid_loss,valid_accuracy,_,_=evaluate(model,valid_loader,loss_function,DEVICE)
    history.append((epoch,train_loss,valid_loss,train_accuracy,valid_accuracy))
    if valid_loss<best_validation:
        best_validation=valid_loss
        best_state={name:tensor.detach().cpu().clone() for name,tensor in model.state_dict().items()}
    if epoch==1 or epoch%20==0:
        print(f"epoch={epoch:3d} train_loss={train_loss:.4f} valid_loss={valid_loss:.4f} train_acc={train_accuracy:.3f} valid_acc={valid_accuracy:.3f}")

model.load_state_dict(best_state)
valid_loss,valid_accuracy,valid_probability,valid_target=evaluate(model,valid_loader,loss_function,DEVICE)
print("best validation loss:",best_validation,"restored accuracy:",valid_accuracy)


epoch=  1 train_loss=0.7269 valid_loss=0.7084 train_acc=0.375 valid_acc=0.438
epoch= 20 train_loss=0.1479 valid_loss=0.1128 train_acc=0.969 valid_acc=0.979
epoch= 40 train_loss=0.1151 valid_loss=0.0895 train_acc=0.974 valid_acc=0.979
epoch= 60 train_loss=0.1050 valid_loss=0.0788 train_acc=0.974 valid_acc=0.979
epoch= 80 train_loss=0.0985 valid_loss=0.0734 train_acc=0.974 valid_acc=0.979
epoch=100 train_loss=0.0932 valid_loss=0.0706 train_acc=0.974 valid_acc=0.979
best validation loss: 0.06919071823358536 restored accuracy: 0.9791666865348816


## 5. Metrics and inference

Probability threshold 0.5 is a policy choice. Metrics should reflect the application's error costs, especially for imbalanced or high-risk tasks.


In [13]:
def binary_metrics(target,probability,threshold=.5):
    y=target.numpy().astype(int).ravel(); pred=(probability.numpy().ravel()>=threshold).astype(int)
    tp=int(np.sum((pred==1)&(y==1))); tn=int(np.sum((pred==0)&(y==0)))
    fp=int(np.sum((pred==1)&(y==0))); fn=int(np.sum((pred==0)&(y==1)))
    precision=tp/(tp+fp) if tp+fp else 0.; recall=tp/(tp+fn) if tp+fn else 0.
    f1=2*precision*recall/(precision+recall) if precision+recall else 0.
    return {"TP":tp,"TN":tn,"FP":fp,"FN":fn,"accuracy":(tp+tn)/len(y),"precision":precision,"recall":recall,"F1":f1}

print(binary_metrics(valid_target,valid_probability))

new_points=torch.tensor([[-2.,-1.],[0.,0.],[2.,1.]],dtype=torch.float32,device=DEVICE)
model.eval()
with torch.no_grad(): new_probability=torch.sigmoid(model(new_points)).cpu().numpy().ravel()
for point,probability in zip(new_points.cpu().numpy(),new_probability):
    print(f"point={point} probability(class 1)={probability:.3f} prediction={int(probability>=.5)}")


{'TP': 20, 'TN': 27, 'FP': 0, 'FN': 1, 'accuracy': 0.9791666666666666, 'precision': 1.0, 'recall': 0.9523809523809523, 'F1': 0.975609756097561}
point=[-2. -1.] probability(class 1)=0.002 prediction=0
point=[0. 0.] probability(class 1)=0.854 prediction=1
point=[2. 1.] probability(class 1)=0.997 prediction=1


# Part II — Mock sentence classification

## 6. Dataset and Bag-of-Words features

The NLP pipeline is

```text
sentence → tokenize → training vocabulary → Bag-of-Words vector → PyTorch MLP
```

Only training sentences are used to build the vocabulary. The data is small and artificial, intended for teaching workflow rather than measuring real sentiment quality.


In [15]:
train_sentences=[
"the lecture was clear and useful","I enjoyed the practical examples","the explanation was easy to follow",
"excellent lesson and helpful exercises","the notebook is detailed and clear","I learned a lot from this class",
"the examples were interesting","good pacing and useful code","the lecture was confusing and slow",
"I disliked the unclear explanation","the examples were boring","poor lesson with unhelpful exercises",
"the notebook is difficult and messy","I learned nothing from this class","the code was confusing","bad pacing and unclear slides"]
train_labels=np.array([1]*8+[0]*8,dtype="float32").reshape(-1,1)
valid_sentences=["clear examples and useful exercises","interesting and helpful lecture",
"confusing notebook and poor explanation","boring examples with bad pacing","the code was clear","the lesson was messy"]
valid_labels=np.array([1,1,0,0,1,0],dtype="float32").reshape(-1,1)

def tokenize(s): return re.findall(r"[a-z]+(?:'[a-z]+)?",s.lower())
counts=Counter(token for sentence in train_sentences for token in tokenize(sentence))
vocab={token:index for index,token in enumerate(sorted(counts))}
def vectorize(sentences):
    matrix=np.zeros((len(sentences),len(vocab)),dtype="float32")
    for row,sentence in enumerate(sentences):
        for token in tokenize(sentence):
            if token in vocab: matrix[row,vocab[token]]=1
    return matrix

X_text_train=torch.tensor(vectorize(train_sentences)); y_text_train=torch.tensor(train_labels)
X_text_valid=torch.tensor(vectorize(valid_sentences)); y_text_valid=torch.tensor(valid_labels)
text_train_loader=DataLoader(TensorDataset(X_text_train,y_text_train),batch_size=4,shuffle=True,generator=torch.Generator().manual_seed(SEED))
text_valid_loader=DataLoader(TensorDataset(X_text_valid,y_text_valid),batch_size=6)
print("vocabulary size:",len(vocab),"training tensor:",X_text_train.shape)


vocabulary size: 45 training tensor: torch.Size([16, 45])


## 7. Build and train the text MLP

The hidden layer learns combinations of word-presence features. Dropout randomly masks hidden activations during training and is automatically disabled by `model.eval()`.


In [17]:
torch.manual_seed(SEED)
text_model=nn.Sequential(
    nn.Linear(len(vocab),16),
    nn.ReLU(),
    nn.Dropout(.15),
    nn.Linear(16,1),
).to(DEVICE)
text_loss=nn.BCEWithLogitsLoss(); text_optimizer=torch.optim.Adam(text_model.parameters(),lr=.02,weight_decay=1e-4)

for epoch in range(1,151):
    train_loss,train_acc=train_one_epoch(text_model,text_train_loader,text_loss,text_optimizer,DEVICE)
    valid_loss,valid_acc,_,_=evaluate(text_model,text_valid_loader,text_loss,DEVICE)
    if epoch==1 or epoch%30==0:
        print(f"epoch={epoch:3d} train_loss={train_loss:.4f} valid_loss={valid_loss:.4f} train_acc={train_acc:.3f} valid_acc={valid_acc:.3f}")


epoch=  1 train_loss=0.7048 valid_loss=0.6653 train_acc=0.500 valid_acc=0.833
epoch= 30 train_loss=0.0021 valid_loss=0.0135 train_acc=1.000 valid_acc=1.000
epoch= 60 train_loss=0.0004 valid_loss=0.0179 train_acc=1.000 valid_acc=1.000
epoch= 90 train_loss=0.0004 valid_loss=0.0257 train_acc=1.000 valid_acc=1.000
epoch=120 train_loss=0.0011 valid_loss=0.0066 train_acc=1.000 valid_acc=1.000
epoch=150 train_loss=0.0008 valid_loss=0.0102 train_acc=1.000 valid_acc=1.000


## 8. Sentence inference and error cases

`predict_sentences` performs the same preprocessing used during training. Inputs containing no known vocabulary items produce an all-zero vector, so their prediction is largely controlled by learned biases.


In [19]:
@torch.no_grad()
def predict_sentences(sentences,threshold=.5):
    text_model.eval()
    features=torch.tensor(vectorize(sentences),device=DEVICE)
    probabilities=torch.sigmoid(text_model(features)).cpu().numpy().ravel()
    return [(s,float(p),int(p>=threshold),int(features[i].sum().item())) for i,(s,p) in enumerate(zip(sentences,probabilities))]

cases=valid_sentences+["not good","clear but boring","fantastic instructor","บทเรียนชัดเจน"]
for sentence,probability,prediction,known in predict_sentences(cases):
    print(f"p={probability:.3f} pred={prediction} known_features={known:2d} | {sentence}")


p=1.000 pred=1 known_features= 5 | clear examples and useful exercises
p=1.000 pred=1 known_features= 4 | interesting and helpful lecture
p=0.000 pred=0 known_features= 5 | confusing notebook and poor explanation
p=0.000 pred=0 known_features= 5 | boring examples with bad pacing
p=0.945 pred=1 known_features= 4 | the code was clear
p=0.004 pred=0 known_features= 4 | the lesson was messy
p=0.947 pred=1 known_features= 1 | not good
p=0.029 pred=0 known_features= 2 | clear but boring
p=0.372 pred=0 known_features= 0 | fantastic instructor
p=0.372 pred=0 known_features= 0 | บทเรียนชัดเจน


## 9. Save and restore parameters

`state_dict` contains model parameters and registered buffers. Architecture code and vocabulary must also be preserved because a state dictionary alone does not explain how to interpret its tensors or text inputs.


In [21]:
# Demonstration uses an in-memory checkpoint to avoid writing an extra model file.
checkpoint={name:tensor.detach().cpu().clone() for name,tensor in text_model.state_dict().items()}
restored_model=nn.Sequential(nn.Linear(len(vocab),16),nn.ReLU(),nn.Dropout(.15),nn.Linear(16,1)).to(DEVICE)
restored_model.load_state_dict(checkpoint); restored_model.eval()
with torch.no_grad():
    original=torch.sigmoid(text_model.eval()(X_text_valid.to(DEVICE)))
    restored=torch.sigmoid(restored_model(X_text_valid.to(DEVICE)))
print("maximum restored prediction difference:",(original-restored).abs().max().item())
assert torch.allclose(original,restored)


maximum restored prediction difference: 0.0


## Practical checklist

Before considering a training run complete, verify:

- deterministic seeds where feasible;
- no train/validation leakage;
- correct tensor shapes and dtypes;
- logits supplied to the chosen loss;
- model and data on the same device;
- `train()` for training and `eval()` for evaluation;
- validation-based model selection;
- task-appropriate metrics and threshold;
- preprocessing and vocabulary saved with the model;
- error analysis on challenging and out-of-domain cases.


## Exercises

1. Change hidden widths and report parameter count plus validation performance.
2. Compare Adam with SGD under at least three learning rates.
3. Add `BatchNorm1d` to the numerical classifier and explain train/eval behavior.
4. Remove dropout from the text model and compare several seeds.
5. Add bigram features to handle `not good` without changing the network code.
6. Introduce class imbalance and use `pos_weight` in `BCEWithLogitsLoss`.
7. Save a complete checkpoint containing model state, vocabulary, threshold, and metadata.


## Summary

- PyTorch layers make model composition concise.
- DataLoader provides shuffled mini-batches during training and deterministic validation batches.
- `BCEWithLogitsLoss` consumes logits; sigmoid is reserved for probabilities.
- A reusable loop separates training and evaluation behavior.
- The same workflow supports geometric and text features.
- Framework automation does not replace careful data splitting, evaluation, error analysis, or reproducible preprocessing.
